# WP3 — YOLOv8 Agent Chaussée (D20 / D40)

**Training: NOT YET EXECUTED** until the cells below complete on a **CUDA GPU** (Tesla T4).

This notebook does **not** contain Precision, Recall, mAP, F1, FPS, or model-size numbers.
Metrics are written to `training/reports/wp3/` from Ultralytics after a real run.
Do not paste placeholder accuracy into markdown.

| Rule | Value |
| --- | --- |
| Agent | WP3 pavement only |
| Classes | 0=D20 alligator, 1=D40 pothole |
| Depth | Forbidden |
| Split | Existing seed-42 hash split — **do not re-split** |
| Expected counts | train 7380 / val 1581 / test 1582 / 17160 boxes |
| YAML | `training/config/pavement.yaml` only |
| CPU | **Forbidden** — stop if CUDA is missing |
| RDD zip | **Do NOT wget** the 13.3GB dump |
| WP2 / WP4 | Out of scope |

Repo: `TrabelsiAmin/Road-damage-project` branch `feature/road-damage-ai-improvements`.
Guide: `docs/wp3-training.md`.


## 1. Env + GPU

Runtime → Change runtime type → **GPU (T4)**. Python, `nvidia-smi`, PyTorch CUDA.
Install **only** `training/requirements.txt`. Never CPU-train.


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

print("python", sys.version)
if shutil.which("nvidia-smi") is None:
    raise SystemExit("CUDA GPU required for WP3 training.")
print(subprocess.check_output(["nvidia-smi"], text=True))


Clone or locate the repo, then `pip install -r training/requirements.txt` (no extra CUDA/CPU torch pin).


In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/TrabelsiAmin/Road-damage-project.git"
BRANCH = "feature/road-damage-ai-improvements"

cwd = Path.cwd()
if (cwd / "training" / "config" / "pavement.yaml").exists():
    REPO_ROOT = cwd
elif (cwd / "config" / "pavement.yaml").exists():
    REPO_ROOT = cwd.parent
else:
    dest = Path("/content/Road-damage-project")
    if not (dest / "training" / "config" / "pavement.yaml").exists():
        subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(dest)])
    REPO_ROOT = dest

TRAINING = (REPO_ROOT / "training").resolve()
os.chdir(TRAINING)
sys.path.insert(0, str(TRAINING))
print("REPO_ROOT", REPO_ROOT.resolve())
print("TRAINING", TRAINING)

# pip only repo training deps (ultralytics pulls a CUDA torch on Colab GPU runtimes)
%pip install -r requirements.txt

import torch
print("torch", torch.__version__)
print("cuda_built", torch.version.cuda)
print("cuda_available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("CUDA GPU required for WP3 training.")
print("device", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print("total_memory_GB", round(props.total_memory / 1024**3, 2))
print("T4_batch_policy", "16 primary, OOM fallback 8")
try:
    import ultralytics
    print("ultralytics", ultralytics.__version__)
except Exception as exc:
    raise SystemExit(f"ultralytics import failed: {exc}")


## 2. Repo / branch check

Expected WP3 files must exist. `training/config/pavement.yaml` names **0: D20**, **1: D40**.
Processed JPEG splits are gitignored — missing images is a data issue, not a reason to invent paths.


In [ ]:
import json, subprocess
from src.verify_wp3_dataset import check_wp3_repo, EXPECTED_COUNTS

try:
    branch = subprocess.check_output(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=str(REPO_ROOT), text=True
    ).strip()
except Exception as exc:
    branch = f"UNKNOWN ({exc})"
print("branch", branch)
if branch not in {"feature/road-damage-ai-improvements", "HEAD"}:
    print("WARNING: expected branch feature/road-damage-ai-improvements")

repo_report = check_wp3_repo(REPO_ROOT)
(TRAINING / "reports" / "wp3").mkdir(parents=True, exist_ok=True)
(TRAINING / "reports" / "wp3" / "repo_check.json").write_text(json.dumps(repo_report, indent=2))
print(json.dumps(repo_report, indent=2)[:4000])
if repo_report["status"] != "OK":
    raise SystemExit("WP3 repo files missing. Do not train.")
print("expected_split_counts", EXPECTED_COUNTS)
print("processed_splits", repo_report.get("processed_splits"))
print("yaml_names", repo_report.get("yaml_names"))
assert repo_report.get("yaml_names", {}).get("0") == "D20"
assert repo_report.get("yaml_names", {}).get("1") == "D40"


## 3. Dataset verify (D20 / D40 only)

If prepared JPEGs are missing, **stop**. Recover via Drive mount or `convert_rdd_voc` + `prepare_dataset`
from an **already-extracted** RDD. **Do NOT wget** the 13.3GB `RDD2022_released_through_CRDDC2022.zip`.

Class IDs must be only 0 and 1. No D00/D10/D50/D60/D90 in WP3 labels.
When images are present, require train/val/test **7380 / 1581 / 1582**.


In [ ]:
import os, json, zipfile
from pathlib import Path
from src.verify_wp3_dataset import (
    locate_pavement_root,
    verify,
    DATA_RECOVERY_INSTRUCTIONS,
    EXPECTED_COUNTS,
)

# Optional Drive mount (no-op outside Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped:", exc)

print(DATA_RECOVERY_INSTRUCTIONS)

DRIVE_PAVEMENT = os.environ.get("WP3_PAVEMENT_ROOT", "")
ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/tariqmap/pavement_wp3_splits.zip"),
    Path("/content/pavement_wp3_splits.zip"),
]
UNZIP_DEST = Path("/content/wp3_data")

root = Path(DRIVE_PAVEMENT) if DRIVE_PAVEMENT else locate_pavement_root()
if root is None or not (Path(root) / "images" / "train").is_dir():
    for z in ZIP_CANDIDATES:
        if z.exists():
            print("unzipping processed splits", z)
            UNZIP_DEST.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(UNZIP_DEST)
            for cand in [UNZIP_DEST / "pavement", UNZIP_DEST]:
                if (cand / "images" / "train").is_dir():
                    root = cand
                    break
            break

if root is None:
    raise SystemExit(DATA_RECOVERY_INSTRUCTIONS)

WP3_ROOT = Path(root).resolve()
os.environ["WP3_PAVEMENT_ROOT"] = str(WP3_ROOT)
print("WP3_ROOT", WP3_ROOT)

yaml_src = TRAINING / "config" / "pavement.yaml"
report = verify(WP3_ROOT, yaml_src, check_expected_counts=True)
(TRAINING / "reports" / "wp3" / "dataset_verify.json").write_text(json.dumps(report, indent=2))
print("jpeg_status", report.get("jpeg_status"))
print("split_images", report.get("split_images"))
print("split_labels", report.get("split_labels"))
print("boxes", report.get("box_counts_by_name"))
print("missing_images", report.get("missing_images"))
print("malformed_labels", report.get("bad_lines"))
print("invalid_boxes", report.get("invalid_boxes"))
print("class_ids_seen", report.get("class_ids_seen"))
print("small_object_est_640", json.dumps(report.get("small_object_est_640"), indent=2)[:2000])
if report["status"] != "OK":
    raise SystemExit(
        "WP3 dataset verification FAILED. Do not train. Do not re-split. "
        + DATA_RECOVERY_INSTRUCTIONS
    )
print("OK expected", EXPECTED_COUNTS)


## 4. Verify YOLO YAML path

Use **`training/config/pavement.yaml`**. Rewrite only `path` to the absolute dataset root.
Do not invent class names or split keys.


In [ ]:
from src.verify_wp3_dataset import write_colab_yaml

yaml_src = TRAINING / "config" / "pavement.yaml"
print("canonical_yaml", yaml_src)
print(yaml_src.read_text())
assert yaml_src.is_file(), "training/config/pavement.yaml missing"
text = yaml_src.read_text()
assert "0: D20" in text and "1: D40" in text
assert "D00" not in text.split("names:")[-1]
assert "D10" not in text.split("names:")[-1]
assert "D50" not in text.split("names:")[-1]

COLAB_YAML = TRAINING / "reports" / "wp3" / "pavement.colab.yaml"
write_colab_yaml(WP3_ROOT, COLAB_YAML)
print("--- colab yaml (path rewritten, names unchanged) ---")
print(COLAB_YAML.read_text())


## 5. Baseline — YOLOv8n, imgsz=640, epochs=100, seed=42, SGD, lr0=0.01

Tesla T4 ~15 GB: **batch 16**, OOM fallback **8**. Never Ultralytics auto-batch, never CPU.

Output: `runs/wp3_baseline/` with `weights/best.pt`, `weights/last.pt`, `results.csv`,
plots, confusion matrix, PR curves (`plots=True`).


In [ ]:
from src.train import train_t4_safe, T4_BATCH, T4_BATCH_OOM_FALLBACK

print("T4_BATCH", T4_BATCH, "OOM_FALLBACK", T4_BATCH_OOM_FALLBACK)
used_batch = train_t4_safe(
    agent="pavement",
    data_config=COLAB_YAML,
    weights="yolov8n.pt",
    epochs=100,
    imgsz=640,
    seed=42,
    aug_config=TRAINING / "config" / "augmentation.yaml",
    project=str(TRAINING / "runs"),
    allow_cpu=False,
    finetune=False,
    run_name="wp3_baseline",
    optimizer="SGD",
    lr0=0.01,
    exist_ok=False,
)
print("completed_batch", used_batch)
BASE_RUN = TRAINING / "runs" / "wp3_baseline"
BASE_WEIGHTS = BASE_RUN / "weights" / "best.pt"
print("best.pt", BASE_WEIGHTS, BASE_WEIGHTS.exists())
print("last.pt", (BASE_RUN / "weights" / "last.pt").exists())
print("results.csv", (BASE_RUN / "results.csv").exists())


## 6. Evaluate the **fixed test** split only

Precision / Recall / mAP50 / mAP50-95 and per-class D20 / D40 come from Ultralytics JSON.
No hardcoded metrics in this markdown.


In [ ]:
from src.eval_wp3 import evaluate_wp3
import json

BASE_WEIGHTS = TRAINING / "runs" / "wp3_baseline" / "weights" / "best.pt"
base_eval = evaluate_wp3(
    weights=BASE_WEIGHTS,
    data=COLAB_YAML,
    split="test",
    imgsz=640,
    output=TRAINING / "reports" / "wp3" / "baseline" / "metrics.json",
    run_name="wp3_baseline",
)
print("status", base_eval["status"])
print("reason", base_eval.get("reason"))
metrics = base_eval.get("metrics")
if metrics:
    print("precision", metrics.get("precision"))
    print("recall", metrics.get("recall"))
    print("mAP50", metrics.get("mAP50"))
    print("mAP50-95", metrics.get("mAP50-95"))
    print("per_class", json.dumps(metrics.get("per_class"), indent=2))
    print("ultralytics_results_dict", metrics.get("ultralytics_results_dict"))
else:
    print("metrics", None, "(NOT_RUN — no fabricated P/R/mAP)")


## 7. Small-object analysis (especially D40)

**Hypotheses** (not detector scores):

1. D40 potholes include more tiny boxes than D20, so they are harder at `imgsz=640`.
2. If that is true, D40 AP50 / recall should lag D20 unless the improvement run helps small boxes.

**Measured now:** ground-truth size bins. **Measured later:** per-class AP from the test JSON above.
If model metrics are missing, status is NOT_RUN — that is not a zero.


In [ ]:
from src.eval_small_objects import analyze_root
import json

so = analyze_root(WP3_ROOT, split=None)
(TRAINING / "reports" / "wp3" / "small_objects_gt.json").write_text(json.dumps(so, indent=2))
print("note", so.get("note"))
for split, block in so.get("splits", {}).items():
    print("====", split, block.get("status"), "====")
    print(json.dumps(block.get("per_class"), indent=2))

ds_report = TRAINING / "reports" / "dataset_report.json"
if ds_report.is_file():
    raw = json.loads(ds_report.read_text())
    print("VOC_small_object_note (dataset measurement, not mAP):",
          raw.get("voc_extract", {}).get("small_objects_area_lt_32px"))

metrics_path = TRAINING / "reports" / "wp3" / "baseline" / "metrics.json"
if metrics_path.is_file():
    ev = json.loads(metrics_path.read_text())
    print("model_eval_status", ev.get("status"))
    per = (ev.get("metrics") or {}).get("per_class") if ev.get("metrics") else None
    print("measured_per_class_vs_hypothesis", per)
else:
    print("measured_per_class_vs_hypothesis", "NOT_RUN")


## 8. Improvement experiment — YOLOv8s imgsz=640 (T4-suitable)

Same **fixed test** set as baseline. Compare measured JSON only.
YOLOv8s/960 is skipped unless free VRAM is clearly above T4 640-headroom.


In [ ]:
import json
from pathlib import Path
from src.train import train_t4_safe
from src.eval_wp3 import evaluate_wp3
from evaluation.compare_wp3_models import compare

print("improvement: YOLOv8s / 640 / 100 / seed 42 / SGD / lr0=0.01 / T4 batch 16→8")
train_t4_safe(
    agent="pavement",
    data_config=COLAB_YAML,
    weights="yolov8s.pt",
    epochs=100,
    imgsz=640,
    seed=42,
    aug_config=TRAINING / "config" / "augmentation.yaml",
    project=str(TRAINING / "runs"),
    allow_cpu=False,
    finetune=False,
    run_name="wp3_expB_s640",
    optimizer="SGD",
    lr0=0.01,
    exist_ok=False,
)
IMP_WEIGHTS = TRAINING / "runs" / "wp3_expB_s640" / "weights" / "best.pt"
imp_eval = evaluate_wp3(
    weights=IMP_WEIGHTS,
    data=COLAB_YAML,
    split="test",
    imgsz=640,
    output=TRAINING / "reports" / "wp3" / "wp3_expB_s640" / "metrics.json",
    run_name="wp3_expB_s640",
)
print("improved_status", imp_eval["status"])
print("improved_metrics_present", bool(imp_eval.get("metrics")))

cmp = compare(
    TRAINING / "reports" / "wp3" / "baseline" / "metrics.json",
    TRAINING / "reports" / "wp3" / "wp3_expB_s640" / "metrics.json",
    TRAINING / "reports" / "wp3" / "compare.json",
)
print(cmp["markdown_table"])
print("declare_better", cmp["declare_better"])
print(cmp["selection_note"])


## 9. Best-model selection (criteria, not largest backbone)

Rank measured test JSON: **D40 AP50**, then **D20 AP50**, then **mAP50**, then smaller file.
Copy the winner to `reports/wp3/artifacts/best.pt`. If nothing is `status=OK`, do not write fake weights.


In [ ]:
from evaluation.select_wp3_best import select_best
from pathlib import Path

sel = select_best(
    [
        {
            "name": "wp3_baseline",
            "metrics_path": TRAINING / "reports" / "wp3" / "baseline" / "metrics.json",
            "weights_path": TRAINING / "runs" / "wp3_baseline" / "weights" / "best.pt",
        },
        {
            "name": "wp3_expB_s640",
            "metrics_path": TRAINING / "reports" / "wp3" / "wp3_expB_s640" / "metrics.json",
            "weights_path": TRAINING / "runs" / "wp3_expB_s640" / "weights" / "best.pt",
        },
    ],
    TRAINING / "reports" / "wp3" / "artifacts" / "best.pt",
)
print("selection_status", sel["status"], "selected", sel.get("selected"))
BEST_PT = TRAINING / "reports" / "wp3" / "artifacts" / "best.pt"
print("artifacts/best.pt exists", BEST_PT.is_file())


## 10. TFLite export — only if a real `best.pt` exists

Do **not** claim the mobile runner is validated unless this file exists **and**
`src.verify_tflite --agent pavement` reports OK **and** a device test was run (it has not).


In [ ]:
from src.export import export_tflite
from src.verify_tflite import inspect_tflite
import json
from pathlib import Path

BEST_PT = TRAINING / "reports" / "wp3" / "artifacts" / "best.pt"
if not BEST_PT.is_file():
    for fallback in (
        TRAINING / "runs" / "wp3_expB_s640" / "weights" / "best.pt",
        TRAINING / "runs" / "wp3_baseline" / "weights" / "best.pt",
    ):
        if fallback.is_file():
            BEST_PT = fallback
            break

export_dir = TRAINING / "reports" / "wp3" / "export"
tflite_path = export_dir / "pavement.tflite"
mobile_validated = False

if not BEST_PT.is_file():
    print("TFLite skipped: no real best.pt (training NOT YET EXECUTED)")
    verify = inspect_tflite(tflite_path, expected_classes=2)
else:
    try:
        tflite_path = export_tflite(
            weights_path=BEST_PT,
            agent="pavement",
            output_dir=export_dir,
            precision="float16",
            imgsz=640,
        )
        print("exported", tflite_path, "bytes", tflite_path.stat().st_size)
    except SystemExit as exc:
        print("export NOT_RUN:", exc)
    verify = inspect_tflite(tflite_path, expected_classes=2)

(TRAINING / "reports" / "wp3" / "tflite_verify.json").write_text(json.dumps(verify, indent=2))
print(json.dumps(verify, indent=2))
if verify.get("status") == "OK" and tflite_path.is_file():
    print("tflite_file_exists", True)
    print("mobile_validated", False, "(on-device test was not run in this notebook)")
else:
    print("mobile_validated", False)


## 11. Artefact directory

Collect `best.pt`, `results.csv`, metrics JSON, plots, confusion, PR, TFLite if any, and `pavement.yaml`.
Zip is optional. **Do not git-commit** weights, TFLite, or datasets (`*.pt` / `*.tflite` / `training/data/` are gitignored).


In [ ]:
from src.pack_wp3_artifacts import stage_artifacts, pack
from pathlib import Path

artifact_dir = TRAINING / "reports" / "wp3" / "artifacts"
selected = artifact_dir / "best.pt"
staged = stage_artifacts(
    TRAINING,
    artifact_dir,
    selected_best=selected if selected.is_file() else None,
)
print("stage_status", staged["status"])
print("copied", staged.get("copied"))

# Optional zip for Colab download — not committed
zip_manifest = pack(TRAINING, TRAINING / "wp3_training_artifacts.zip")
print("zip_status", zip_manifest["status"], "files", zip_manifest["file_count"])
print("Download the zip from Colab if needed. Do not git-commit *.pt / *.tflite / this zip / datasets.")


## 12. Docs, limitations, next steps

Written in-repo (do not invent metrics here):

- `docs/wp3-training.md` — dataset, config, T4 hardware, commands, experiments, limitations
- `docs/traceability.md` — WP3 rows only; WP2/WP4 stay separate agents
- `training/reports/METRICS.md` — **NOT RUN** until Colab JSON has `"status": "OK"`

**Training: NOT YET EXECUTED** if you stopped before the GPU train cells, or if this notebook
is opened on a CPU runtime.

Next: copy `reports/wp3/**/*.json` back to git (not weights). Then teacher/KD, then WP2/WP4.


## Optional: KD stub (does not train, does not invent metrics)

`src.distill` remains a stub until a teacher checkpoint exists.


In [ ]:
from src.distill import plan
from pathlib import Path

kd = plan(
    agent="pavement",
    teacher=TRAINING / "runs" / "wp3_expB_s640" / "weights" / "best.pt",
    student=Path("yolov8n.pt"),
    data=COLAB_YAML,
    output=TRAINING / "reports" / "wp3" / "kd_plan.json",
)
print("KD status", kd["status"], "metrics", kd["metrics"])
print(kd["reason"])
